## Load NAB time series 

## Step 1 - Load CPU dataset and attach anomaly labels

In this step, the raw `cpu_utilization_asg_misconfiguration.csv` file from the NAB `realKnownCause` folder is loaded into a dataframe. The corresponding anomaly timestamps are retrieved from `combined_labels.json` (using the key `realKnownCause/cpu_utilization_asg_misconfiguration.csv`) and converted into a binary `is_anomaly` column. The `timestamp` column is parsed into a proper datetime type and the rows are sorted chronologically so that each record reflects the correct order in time. Finally, a class-balance table is produced to confirm the number and proportion of normal vs anomalous points before moving into schema standardisation.


### A) Load raw cpu_utilization_asg_misconfiguration.csv dataset

In [36]:
# Imports
import pandas as pd
import numpy as np


In [37]:
import os

# Path to Numeta Anomoly Benchmark (NAB) data folder
base_path = "../data/NAB-master/data/realKnownCause/"

# Load file
file_path = os.path.join(base_path, "cpu_utilization_asg_misconfiguration.csv")
df = pd.read_csv(file_path)

df.head()


,timestamp,value
0,2014-05-14 01:14:00,85.835
1,2014-05-14 01:19:00,88.167
2,2014-05-14 01:24:00,44.595
3,2014-05-14 01:29:00,56.282
4,2014-05-14 01:34:00,36.534


### B) Load Anomaly Labels (JSON)

In [38]:
import json  

# Where the labels file is
label_path = "../data/NAB-master/labels/combined_labels.json"

# Open the file and read its contents
with open(label_path, "r") as label_file:      # "r" = read mode, label_file = name for the opened file
    labels = json.load(label_file)             # turn the JSON into a Python dictionary called 'labels'

# Show the first 10 dataset keys names to confirm it loaded correctly
list(labels.keys())[:10]

['artificialNoAnomaly/art_daily_no_noise.csv',
 'artificialNoAnomaly/art_daily_perfect_square_wave.csv',
 'artificialNoAnomaly/art_daily_small_noise.csv',
 'artificialNoAnomaly/art_flatline.csv',
 'artificialNoAnomaly/art_noisy.csv',
 'artificialWithAnomaly/art_daily_flatmiddle.csv',
 'artificialWithAnomaly/art_daily_jumpsdown.csv',
 'artificialWithAnomaly/art_daily_jumpsup.csv',
 'artificialWithAnomaly/art_daily_nojump.csv',
 'artificialWithAnomaly/art_increase_spike_density.csv']

### C) Extract Labels for Selected Dataset

In [39]:
# Select the correct dataset key from the labels dictionary
dataset_key = "realKnownCause/cpu_utilization_asg_misconfiguration.csv"

# Extract the list of anomaly timestamps for this dataset
cpu_label_times = labels[dataset_key]

# Turn the list of timestamp strings into a DataFrame
cpu_label_times_df = pd.DataFrame(
    {"timestamp": pd.to_datetime(cpu_label_times)}
)

# Display the timestamps
cpu_label_times_df.head()

,timestamp
0,2014-07-12 02:04:00
1,2014-07-14 21:44:00


### D) Add Labels to DataFrame

In [40]:
# Converting the label timestamps into datetime objects 
cpu_label_times = pd.to_datetime(labels[dataset_key])

# Ensure timestamp is datetime and sorted
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

# Create a new column: 1 if the timestamp is an anomaly, 0 otherwise
df["is_anomaly"] = df["timestamp"].isin(cpu_label_times).astype(int)

# See the updated dataframe
df.head()


,timestamp,value,is_anomaly
0,2014-05-14 01:14:00,85.835,0
1,2014-05-14 01:19:00,88.167,0
2,2014-05-14 01:24:00,44.595,0
3,2014-05-14 01:29:00,56.282,0
4,2014-05-14 01:34:00,36.534,0


### E) Class Balance

In [41]:
# Class balance for the is_anomaly column
anomaly_balance = (
    df["is_anomaly"]
      .value_counts()                            # counts for 0 and 1
      .rename(index={0: "Normal", 1: "Anomaly"}) # rename classes for readability
      .to_frame(name="Count")                    # make it a DataFrame with a 'Count' column
      .reset_index()                             # turn index into a normal column
      .rename(columns={"is_anomaly": "Class"})       
)

# Percentage column
anomaly_balance["Proportion (%)"] = (
    anomaly_balance["Count"] / len(df) * 100
).round(3)

anomaly_balance


,Class,Count,Proportion (%)
0,Normal,18048,99.989
1,Anomaly,2,0.011


## Step 2 - Standardise core columns (`time`, `value`, `is_anomaly`, `case_study`)

In [42]:
# Ensure timestamp is datetime and sorted
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

# Rename to common research project schema
df = df.rename(columns={"timestamp": "time"})

# Add a case_study identifier
df["case_study"] = "cpu"

# Quick check
df[["time", "value", "is_anomaly", "case_study"]].head()


,time,value,is_anomaly,case_study
0,2014-05-14 01:14:00,85.835,0,cpu
1,2014-05-14 01:19:00,88.167,0,cpu
2,2014-05-14 01:24:00,44.595,0,cpu
3,2014-05-14 01:29:00,56.282,0,cpu
4,2014-05-14 01:34:00,36.534,0,cpu


###  Quick verification

In [43]:
df[["time","value","is_anomaly","case_study"]].dtypes

time          datetime64[ns]
value                float64
is_anomaly             int64
case_study            object
dtype: object


In this step, the CPU utilisation dataframe is aligned with the common schema used across all case studies. The `timestamp` column is converted to a proper datetime type, the rows are sorted chronologically, and the column is renamed to `time`. A `case_study` column with the value `"cpu"` is added so that this dataset can later be stacked or filtered alongside the other case studies using the same core columns (`time`, `value`, `is_anomaly`, `case_study`).


## Step 3 - Missing values and Duplicates rows sanity check

### A) Missing values per column


In [44]:
# Count missing values in each column
missing_counts = df.isna().sum()

# Percentage of missing values in each column
missing_percentages = (df.isna().mean() * 100).round(3)

# Combine into a summary table
missing_summary = (
    pd.DataFrame(
        {
            "Column": missing_counts.index,
            "Missing count": missing_counts.values,
            "Missing (%)": missing_percentages.values,
        }
    )
    .sort_values("Missing count", ascending=False)
    .reset_index(drop=True)
)

missing_summary

,Column,Missing count,Missing (%)
0,time,0,0.0
1,value,0,0.0
2,is_anomaly,0,0.0
3,case_study,0,0.0


### B) Duplicate rows summary 

In [45]:
# Duplicate rows summary
duplicate_count = df.duplicated().sum()
total_rows = len(df)

duplicate_summary = pd.DataFrame(
    [
        ("Total rows", total_rows),
        ("Number of duplicate rows", duplicate_count),
        (
            "Duplicate rows (%)",
            round(duplicate_count / total_rows * 100, 3),
        ),
    ],
    columns=["Statistic", "Value"],
)

duplicate_summary


,Statistic,Value
0,Total rows,18050.0
1,Number of duplicate rows,0.0
2,Duplicate rows (%),0.0



The missing-values table confirms that the CPU utilisation dataset has no missing entries in any of the core columns (`time`, `value`, `is_anomaly`, `case_study`). The duplicate-rows summary also shows zero duplicate records. These checks indicate that the raw NAB file for CPU utilisation is already clean at the core-schema level, and no imputation or de-duplication is required before proceeding to feature engineering and split design. Recording these results explicitly keeps the preprocessing pipeline transparent for later write-up.


## Step 4 - Create basic time-based features

In [46]:
# Hour of day (0–23)
df["hour_of_day"] = df["time"].dt.hour

# Day of week (0 = Monday, 6 = Sunday)
df["day_of_week"] = df["time"].dt.dayofweek

# Weekend flag (Saturday = 5, Sunday = 6)
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)

# Quick check
df[["time", "hour_of_day", "day_of_week", "is_weekend"]].head()


,time,hour_of_day,day_of_week,is_weekend
0,2014-05-14 01:14:00,1,2,0
1,2014-05-14 01:19:00,1,2,0
2,2014-05-14 01:24:00,1,2,0
3,2014-05-14 01:29:00,1,2,0
4,2014-05-14 01:34:00,1,2,0


This step derives simple calendar features from the `time` column to provide
additional structure for later analysis and modelling. The timestamp is broken
down into:

- `hour_of_day`: the hour of the day (0–23),
- `day_of_week`: the day index (0 = Monday, …, 6 = Sunday),
- `is_weekend`: an indicator variable that is 1 for Saturday and Sunday, and 0
  for weekdays.

These features do not change the underlying temperature values, but they allow
later models and visualisations to capture patterns that depend on time of day
or weekday/weekend effects.


## Step 5 - Inspect time span and anomaly distribution

### 5A) Anomaly timestamps and values

In [47]:
# Filter to rows labelled as anomalies and keep only columns time and value
anomaly_summary = (
    df.loc[df["is_anomaly"] == 1, ["time", "value"]]
      .sort_values("time")
      .reset_index(drop=True)
       .rename(columns={
          "time": "Anomaly time",
          "value": "CPU utilisation (%)",
      })
)

# Insterting an Anomaly ID column for readability
anomaly_summary.insert(0, "Anomaly ID", [f"Anomaly {i+1}" for i in range(len(anomaly_summary))])

anomaly_summary


,Anomaly ID,Anomaly time,CPU utilisation (%)
0,Anomaly 1,2014-07-12 02:04:00,52.456
1,Anomaly 2,2014-07-14 21:44:00,14.714


### 5B) Overall time span summary

In [48]:
# Time-span statistics for the series
start_time, end_time = df["time"].min(), df["time"].max()
n_rows = len(df)
duration_days = (end_time - start_time).days

# Summary table 
time_span_summary = pd.DataFrame(
    [
        ("Start time", start_time),
        ("End time", end_time),
        ("Number of rows", n_rows),
        ("Total duration (days)", duration_days),
    ],
    columns=["Statistic", "Value"],
)

time_span_summary


,Statistic,Value
0,Start time,2014-05-14 01:14:00
1,End time,2014-07-15 17:19:00
2,Number of rows,18050
3,Total duration (days),62


### Quick internal check of sampling interval

In [49]:
# Compute differences between consecutive timestamps
time_difference = df["time"].diff().dropna()

# Identify the most common time difference
most_common_difference = time_difference.value_counts().idxmax()

most_common_difference


Timedelta('0 days 00:05:00')

Step 5 checks where the labelled anomalies appear in the CPU utilisation series, how long the dataset runs for, and how often measurements were recorded.

The anomaly table lists the two NAB-labelled events with an `Anomaly ID`, the exact timestamps, and the CPU utilisation values (%) at those times. This makes the abnormal events easy to point to later when describing what went wrong, and it also helps when choosing where the training, validation, and test splits should start and end.

A quick sampling check is done by looking at the time gap between one row and the next. The most common gap is **5 minutes**, which confirms that the dataset is mainly recorded at a 5-minute frequency.

The date range summary shows that the CPU series runs from **2014-05-14 01:14:00** to **2014-07-15 17:19:00**, with **18,050** observations over **62 days**. This gives clear context for the size and coverage of the dataset before defining the chronological train, validation, and test splits in the next steps.


## Step 6 - Define chronological train/validation/test boundaries

### 6A) Identify labelled anomaly times for design split

In [50]:
# Accessing labeled anomalies
first_anomaly_time = anomaly_summary["Anomaly time"].min()
second_anomaly_time = anomaly_summary["Anomaly time"].max()

labelled_anomaly_times = pd.DataFrame(
    [
        ("First anomaly", first_anomaly_time),
        ("Second anomaly", second_anomaly_time),
    ],
    columns=["Statistic", "Value"],
)

labelled_anomaly_times


,Statistic,Value
0,First anomaly,2014-07-12 02:04:00
1,Second anomaly,2014-07-14 21:44:00


### 6B) Create hourly summary series

In [51]:
# This hourly view is only used to guide split-boundary selection (the main dataset stays 5-minute).

hourly_mean = (
    df.set_index("time")["value"]
      .resample("1h")              #groups the data into 1-hour bins 
      .mean()                      #takes the average per hour
      .reset_index()
)

hourly_mean.head()


,time,value
0,2014-05-14 01:00:00,49.703600
1,2014-05-14 02:00:00,38.210083
2,2014-05-14 03:00:00,40.346917
3,2014-05-14 04:00:00,35.579833
4,2014-05-14 05:00:00,32.913333


### 6C) Baseline window cutoff sensitivity check (7, 14, 21 days)

In [52]:
# Comparing a few baseline cutoffs to justify the final choice of baseline window used in data.

# Candidate cutoffs (days before the first labelled anomaly)
cutoff_days_options = [7, 14, 21]
baseline_cutoff_rows = []

for days in cutoff_days_options:
    baseline_cutoff_time_option = first_anomaly_time - pd.Timedelta(days=days)  # Includes only hourly values earlier than this timestamp
    baseline_values = hourly_mean.loc[hourly_mean["time"] < baseline_cutoff_time_option, "value"].dropna()   # Hourly CPU utilisation values before the cutoff time

    # Summary stats and reference thresholds the cutoff option
    baseline_cutoff_rows.append(
        {
            "Cutoff (days before first anomaly)": days,
            "Baseline cutoff time": baseline_cutoff_time_option,
            "Baseline hours used": len(baseline_values),
            "Baseline mean CPU (%)": baseline_values.mean(),
            "Baseline std CPU (%)": baseline_values.std(),
            "High baseline threshold (99.5th percentile)": baseline_values.quantile(0.995),
            "Low baseline threshold (5th percentile)": baseline_values.quantile(0.05),
        }
    )

baseline_cutoff_sensitivity = pd.DataFrame(baseline_cutoff_rows).round(3)
baseline_cutoff_sensitivity


,Cutoff (days before first anomaly),Baseline cutoff time,Baseline hours used,Baseline mean CPU (%),Baseline std CPU (%),High baseline threshold (99.5th percentile),Low baseline threshold (5th percentile)
0,7,2014-07-05 02:04:00,1250,37.386,2.965,52.394,33.320
1,14,2014-06-28 02:04:00,1082,37.176,3.007,52.967,33.254
2,21,2014-06-21 02:04:00,914,37.188,3.086,53.648,33.467


#### Step 6C – Baseline window cutoff sensitivity check (7, 14, 21 days)

- **Purpose of this step**
  - The baseline period is defined as the early segment of the CPU utilisation series and is used as a reference for typical behaviour.
  - This reference supports the calculation of two cutoffs that indicate unusually high and unusually low values relative to baseline behaviour.
  - These cutoffs are later used to help locate the shift into a sustained higher/unstable period and the later drop into a sustained lower period.

- **Baseline cutoff options evaluated**
  - Three candidate cutoff windows are compared:
    - **7 days** before the first labelled anomaly
    - **14 days** before the first labelled anomaly
    - **21 days** before the first labelled anomaly

- **What is computed for each cutoff option**
  - All **hourly** CPU utilisation values before the cutoff time are treated as baseline values.
  - The sensitivity table reports:
    - the number of baseline hours included,
    - baseline mean and spread,
    - **High baseline threshold (99.5th percentile)**,
    - **Low baseline threshold (5th percentile)**.

- **Key finding**
  - Baseline mean and both reference thresholds are very similar across the three cutoff choices.
  - This indicates that early CPU behaviour is stable and that the baseline reference is not highly sensitive to the exact cutoff placement within the tested range.

- **Decision**
  - A **7-day cutoff** is selected as the final baseline window because:
    - it retains the most baseline data,
    - it remains before the first labelled anomaly,
    - the thresholds remain consistent compared with longer cutoff windows.
  - This provides a clear and defensible baseline reference for the threshold-based regime checks applied in the next step.


### 6D) Final baseline cutoff (7 days) reference thresholds

In [53]:
# Baseline values are taken from the earlier period in dataset, ending 7 days before the first labelled anomaly.

# Final baseline cutoff time 
baseline_cutoff_time = first_anomaly_time - pd.Timedelta(days=7)

# Baseline values: hourly CPU utilisation values before the cutoff time
baseline_values = hourly_mean.loc[hourly_mean["time"] < baseline_cutoff_time, "value"].dropna()

# Reference thresholds based on baseline behaviour
high_baseline_threshold = baseline_values.quantile(0.995)  # Rare high baseline cutoff
low_baseline_threshold = baseline_values.quantile(0.05)    # Rare low baseline cutoff

# Thresholds summary table
baseline_thresholds_summary = pd.DataFrame(
    [
        ("Baseline cutoff time", baseline_cutoff_time),
        ("Baseline hours used", len(baseline_values)),
        ("Baseline mean CPU (%)", baseline_values.mean()),
        ("Baseline std CPU (%)", baseline_values.std()),
        ("High baseline threshold (99.5th percentile)", high_baseline_threshold),
        ("Low baseline threshold (5th percentile)", low_baseline_threshold),
    ],
    columns=["Statistic", "Value"],
)

# Round off only numeric values in the Value column for display
baseline_thresholds_summary["Value"] = baseline_thresholds_summary["Value"].apply(
    lambda x: round(x, 3) if isinstance(x, (int, float)) else x
)
baseline_thresholds_summary


,Statistic,Value
0,Baseline cutoff time,2014-07-05 02:04:00
1,Baseline hours used,1250
2,Baseline mean CPU (%),37.386
3,Baseline std CPU (%),2.965
4,High baseline threshold (99.5th percentile),52.394
5,Low baseline threshold (5th percentile),33.32


### 6E) Identify behaviour change points and define split boundary timestamps (regime-based boundaries)

In [54]:
# Operation makes use of the baseline thresholds from 6D and an hourly moving median to focus on sustained changes.

# Moving median settings (hourly series)
smoothing_window_hours = 24
min_hours_required = 12    # prevents unstable values at the beginning of the series

# Calculate 24-hour moving median on the hourly series 
hourly_mean["moving_median_24h"] = (
    hourly_mean["value"]
    .rolling(window=smoothing_window_hours, min_periods=min_hours_required)
    .median()
)

# Detect the start of sustained high/unstable period in dataset
high_regime_start_time = hourly_mean.loc[
    hourly_mean["moving_median_24h"] > high_baseline_threshold, "time"
].min()

# Detect the start of sustained low period in dataset (search only after the high period begins)
low_regime_start_time = hourly_mean.loc[
    (hourly_mean["time"] >= high_regime_start_time) &
    (hourly_mean["moving_median_24h"] < low_baseline_threshold),
    "time"
].min()

# Buffers tied to smoothing window (research-aligned rule) 
train_buffer = pd.Timedelta(hours=smoothing_window_hours * 2)      # 48 hours (24h * 2)
validation_buffer = pd.Timedelta(hours=smoothing_window_hours / 2) # 12 hours (0.5 * 24h)

# --- Split boundary timestamps ---
training_end_time = high_regime_start_time - train_buffer
validation_end_time = low_regime_start_time - validation_buffer

# --- Examiner-friendly summary table ---
split_boundary_summary = pd.DataFrame(
    [
        ("Moving median window (hours)", smoothing_window_hours),
        ("High baseline threshold (99.5th percentile)", high_baseline_threshold),
        ("Low baseline threshold (5th percentile)", low_baseline_threshold),
        ("High/unstable period start time", high_regime_start_time),
        ("Low period start time (post-high search)", low_regime_start_time),
        ("Training buffer", train_buffer),
        ("Validation buffer", validation_buffer),
        ("Training end time", training_end_time),
        ("Validation end time", validation_end_time),
        ("Boundary order valid (train < val)", training_end_time < validation_end_time),
    ],
    columns=["Statistic", "Value"],
)

split_boundary_summary


,Statistic,Value
0,Moving median window (hours),24
1,High baseline threshold (99.5th percentile),52.393835
2,Low baseline threshold (5th percentile),33.320371
3,High/unstable period start time,2014-07-12 13:00:00
4,Low period start time (post-high search),2014-07-15 08:00:00
5,Training buffer,2 days 00:00:00
6,Validation buffer,0 days 12:00:00
7,Training end time,2014-07-10 13:00:00
8,Validation end time,2014-07-14 20:00:00
9,Boundary order valid (train < val),True


#### Step 6E – Identify regime change points and define split boundary timestamps (CPU utilisation)

- **Purpose of this step**
  - Identify when the CPU utilisation series enters a sustained **higher/unstable** period and when it later shifts into a sustained **lower** period.
  - Use these change points to define chronological split boundaries:
    - `training_end_time` (end of training period)
    - `validation_end_time` (end of validation period)

- **Method used**
  - The hourly series is smoothed using a **24-hour moving median** (`moving_median_24h`).
  - This smoothing reduces the influence of short spikes and makes sustained level shifts easier to locate.

- **Change-point detection logic**
  - **Start of higher/unstable period**
    - Defined as the earliest timestamp where `moving_median_24h` rises **above** the **high baseline threshold** (99.5th percentile of baseline behaviour).
  - **Start of lower period**
    - Defined as the earliest timestamp where `moving_median_24h` falls **below** the **low baseline threshold** (5th percentile of baseline behaviour),
    - and searched **only after** the higher/unstable period begins to avoid selecting an earlier low fluctuation that is unrelated to the later transition.

- **Why buffers are applied**
  - The moving median at any time point is influenced by values across a broad window (up to roughly a day of neighbouring observations).
  - If a split boundary is placed directly on a detected change point, information from the transition period can influence both sides of the boundary through the smoothing, which weakens separation between phases.

- **Buffer rule for split boundaries (tied to the smoothing window)**
  - With a **24-hour** smoothing window:
    - **Training buffer = 48 hours (2 × 24h)**
      - Training is ended well before the high/unstable period begins.
      - The larger buffer prioritises a cleaner separation so training mostly reflects earlier behaviour rather than the transition period.
    - **Validation buffer = 12 hours (0.5 × 24h)**
      - Validation is placed closer to the later transition so it still contains meaningful change behaviour.
      - The smaller buffer avoids placing the boundary exactly on the detected low-period start while preserving enough validation/test coverage in a short end-of-series tail.

- **Outputs produced**
  - A summary table reporting:
    - smoothing window length,
    - detected change-point timestamps (high/unstable start and low start),
    - buffer sizes,
    - final `training_end_time` and `validation_end_time`,
    - a boundary ordering check confirming `training_end_time < validation_end_time`.

- **Outcome**
  - The boundary timestamps are ready for Step 7, where each 5-minute observation is labelled as `"train"`, `"validation"`, or `"test"` based on these times.



### 6F) Enforce a minimum test duration (CPU-specific split validity)

In [55]:
# Enforce minimum test size using 5-minute points (strict on-grid rule)

min_test_days = 2              # defensible for the short end-of-series regime 
points_per_day = 24 * 60 // 5      # 288 points per day (1 day has: 24 hours × 60 minutes = 1440 minutes) Each pt has 5 mins - So points per day = 1440 ÷ 5 = 288 points 
min_test_points = min_test_days * points_per_day     # 576 points (2 x 288)


series_end_time = df["time"].max()      # Get the last timestamp  
min_test_start_time = df["time"].iloc[-min_test_points]     # locates the time that guarantees a window containing the last 576 rows of the dataset.

# Move validation_end_time earlier if needed
#validation_end_time = min(validation_end_time, min_test_start_time)
validation_end_time = min(validation_end_time, min_test_start_time - pd.Timedelta(minutes=5))

# Check
boundary_check = pd.DataFrame(
    [
        ("Series end time", series_end_time),
        ("Minimum test days (target)", min_test_days),
        ("Minimum test points (strict)", min_test_points),
        ("Minimum test start time (grid)", min_test_start_time),
        ("Training end time", training_end_time),
        ("Validation end time (adjusted)", validation_end_time),
        ("Boundary order valid (train < val)", training_end_time < validation_end_time),
    ],
    columns=["Statistic", "Value"],
)

boundary_check


,Statistic,Value
0,Series end time,2014-07-15 17:19:00
1,Minimum test days (target),2
2,Minimum test points (strict),576
3,Minimum test start time (grid),2014-07-13 17:24:00
4,Training end time,2014-07-10 13:00:00
5,Validation end time (adjusted),2014-07-13 17:19:00
6,Boundary order valid (train < val),True


#### Step 6F - Enforce a minimum test size using the 5-minute sampling grid (CPU-specific split validity)

- **Why this step is necessary**
  - The CPU utilisation series shows its labelled anomalies and the strongest behaviour changes near the end of the timeline.
  - Regime-based split boundaries (Step 6E) can therefore leave a short test segment if the validation-to-test boundary is placed too late.
  - A short test segment weakens evaluation credibility because results become sensitive to a small number of observations.

- **Principle applied**
  - Splits remain **chronological** and **regime-informed**:
    - `training_end_time` remains anchored to the high/unstable regime transition logic from Step 6E.
  - A dataset-specific validity constraint is applied to ensure a defensible evaluation window:
    - the test split must contain at least **2 days of data** under 5-minute sampling.

- **Revision decision (grid-based enforcement)**
  - The minimum test requirement is enforced using the dataset’s fixed **5-minute sampling grid** rather than only using clock-time differences.
  - Under 5-minute sampling:
    - **288 observations per day** (24 × 60 ÷ 5)
    - **576 observations for 2 days**
  - This approach ensures the test set contains a sufficient number of observations even when the start/end timestamps fall slightly short of an exact 48-hour difference due to sampling alignment.

- **What is adjusted**
  - `validation_end_time` is moved **earlier if required** so that the final test segment includes at least **576 observations**.
  - `training_end_time` is not modified in this step.

- **Outcome**
  - The CPU split design retains the regime-based intent from Step 6E while guaranteeing a test segment with sufficient coverage for reliable evaluation.


## Step 7 - Assign split labels and summarise 

#### 7A) Assign split labels to all rows based on the boundary times

In [56]:
# Splits are chronological and applied to the full 5-minute dataframe using boundary timestamps from Step 6F.

def assign_split(time_value):
    if time_value <= training_end_time:
        return "train"
    elif time_value <= validation_end_time:
        return "validation"
    else:
        return "test"

df["split"] = df["time"].apply(assign_split)

# Split count table
split_counts = (
    df["split"]
    .value_counts()
    .rename_axis("Split")
    .reset_index(name="Rows")
)

# Keep a consistent split order
split_order = ["train", "validation", "test"]
split_counts["Split"] = pd.Categorical(split_counts["Split"], categories=split_order, ordered=True)
split_counts = split_counts.sort_values("Split").reset_index(drop=True)

split_counts


,Split,Rows
0,train,16558
1,validation,916
2,test,576


#### 7B) Split duration summary (5-minute series)

In [57]:
# Summarises time coverage per split on CPU dataset to confirm the boundaries produce sensible partitions.

split_time_summary = (
    df.groupby("split")["time"]
      .agg(Start_time="min", End_time="max", Rows="count")
      .reset_index()
)

# Convert start and end time into a duration measured in days
split_time_summary["Duration_days"] = (
    (split_time_summary["End_time"] - split_time_summary["Start_time"])
    .dt.total_seconds() / (60 * 60 * 24)
).round(2)

# Add the proportion of rows from entire dataset of each split
total_rows = len(df)
split_time_summary["Proportion (%)"] = (split_time_summary["Rows"] / total_rows * 100).round(2)

# Desired row order: train -> validation -> test
split_order = ["train", "validation", "test"]
split_time_summary["split"] = pd.Categorical(split_time_summary["split"], categories=split_order, ordered=True)
split_time_summary = (
    split_time_summary.sort_values("split")
                     .reset_index(drop=True)
)

split_time_summary


,split,Start_time,End_time,Rows,Duration_days,Proportion (%)
0,train,2014-05-14 01:14:00,2014-07-10 12:59:00,16558,57.49,91.73
1,validation,2014-07-10 13:04:00,2014-07-13 17:19:00,916,3.18,5.07
2,test,2014-07-13 17:24:00,2014-07-15 17:19:00,576,2.00,3.19


#### Step 7 - Assign split labels and summarise (CPU utilisation)

Step 7 assigns a split label (`train`, `validation`, `test`) to every row in the CPU utilisation dataset using the regime-informed boundary timestamps defined in Step 6 (including the minimum test-size constraint).

The `split` column is created by comparing each timestamp to the two boundary times:
- observations up to `training_end_time` are labelled **train**
- observations after `training_end_time` up to `validation_end_time` are labelled **validation**
- observations after `validation_end_time` are labelled **test**

This produces a strictly chronological split design so that training is restricted to earlier behaviour and evaluation is performed on later periods that contain the transition into the high/unstable regime and the labelled anomaly windows. The CPU dataset is short and concentrates its most unusual behaviour near the end of the series, so Step 6 includes a CPU-specific validity rule to ensure that the test split contains sufficient coverage under 5-minute sampling.

The summary tables report, for each split, the start and end times, number of rows, duration in days, anomaly counts, and proportion of the full dataset. These outputs confirm that the split boundaries are ordered correctly (no overlap), that each split is internally time-sorted, and that labelled anomalies fall within the evaluation splits (validation/test), supporting later model selection and final performance reporting under a rare-event setting.


#### 7C) Anomaly summary per split

In [58]:
# Confirms where labelled anomalies fall and how concentrated they are across splits.

split_anomaly_summary = (
    df.groupby("split")
      .agg(
          Rows=("is_anomaly", "size"),
          Anomalies=("is_anomaly", "sum"),
      )
      .reset_index()
)

# Anomaly rate per split
split_anomaly_summary["Anomaly_rate (%)"] = (
    split_anomaly_summary["Anomalies"] / split_anomaly_summary["Rows"] * 100
).round(3)

# Desired row order: train -> validation -> test
split_order = ["train", "validation", "test"]
split_anomaly_summary["split"] = pd.Categorical(split_anomaly_summary["split"], categories=split_order, ordered=True)
split_anomaly_summary = split_anomaly_summary.sort_values("split").reset_index(drop=True)

split_anomaly_summary


,split,Rows,Anomalies,Anomaly_rate (%)
0,train,16558,0,0.000
1,validation,916,1,0.109
2,test,576,1,0.174


## Step 8 - Create split dataframes and run integrity checks

#### 8A) Create split dataframes and run integrity checks

In [59]:
# Keep track of the total number of rows in the full processed NYC taxi dataframe
total_rows_full = len(df)

# Create non-destructive split-specific dataframes using boolean masks
train_df = df.loc[df["split"] == "train"].copy()
validation_df = df.loc[df["split"] == "validation"].copy()
test_df = df.loc[df["split"] == "test"].copy()

# Row counts in each split dataframe
n_train = len(train_df)
n_validation = len(validation_df)
n_test = len(test_df)

# Construct integrity summary to check that no rows are lost
split_size_check = pd.DataFrame(
    [
        ("Full dataset rows", total_rows_full),
        ("Train rows", n_train),
        ("Validation rows", n_validation),
        ("Test rows", n_test),
        ("Train + Validation + Test", n_train + n_validation + n_test),
        ("Split sizes add up", total_rows_full == n_train + n_validation + n_test),
    ],
    columns=["Statistic", "Value"],
)

split_size_check


,Statistic,Value
0,Full dataset rows,18050
1,Train rows,16558
2,Validation rows,916
3,Test rows,576
4,Train + Validation + Test,18050
5,Split sizes add up,True


#### 8B) Integrity checks: chronological order and boundary consistency

In [60]:
# Ensure each split is sorted by time 
train_sorted = train_df["time"].is_monotonic_increasing
validation_sorted = validation_df["time"].is_monotonic_increasing
test_sorted = test_df["time"].is_monotonic_increasing

# dataframes
train_time = train_df["time"]
validation_time = validation_df["time"]
test_time = test_df["time"]

# Boundary checks: train ends before validation starts; validation ends before test starts
boundary_integrity = pd.DataFrame(
    [
        ("Train sorted by time", train_sorted),
        ("Validation sorted by time", validation_sorted),
        ("Test sorted by time", test_sorted),
        ("Train start time", train_time.min()),
        ("Train end time", train_time.max()),
        ("Validation start time", validation_time.min()),
        ("Validation end time", validation_time.max()),
        ("Test start time", test_time.min()),
        ("Test end time", test_time.max()),
        ("Train ends before validation starts", train_df["time"].max() < validation_df["time"].min()),
        ("Validation ends before test starts", validation_df["time"].max() < test_df["time"].min()),
    ],
    columns=["Check", "Result"],
)

boundary_integrity


,Check,Result
0,Train sorted by time,True
1,Validation sorted by time,True
2,Test sorted by time,True
3,Train start time,2014-05-14 01:14:00
4,Train end time,2014-07-10 12:59:00
5,Validation start time,2014-07-10 13:04:00
6,Validation end time,2014-07-13 17:19:00
7,Test start time,2014-07-13 17:24:00
8,Test end time,2014-07-15 17:19:00
9,Train ends before validation starts,True


#### Step 8B – Integrity checks for split dataframes (chronology and boundary consistency)

- **Purpose of this step**
  - Confirm that the split-specific dataframes (`train_df`, `validation_df`, `test_df`) are internally consistent and suitable for downstream preprocessing and modelling.
  - Verify that splits are strictly chronological, non-overlapping, and correctly ordered in time.

- **Interpretation of results**
  - All three splits are sorted by time and the boundary checks confirm there is no overlap between splits.
  - The reported start/end times provide a clear view of how much time coverage each split contains and confirm the split design is being applied as intended.

- **Note on boundary timestamps vs observed split end times**
  - Split boundaries are defined using cutoff variables (e.g., `training_end_time`, `validation_end_time`) and are applied using comparison rules such as `time_value <= training_end_time`.
  - Because the CPU series is sampled every 5 minutes, there may be no observation exactly at a boundary timestamp.
  - For this reason, the final timestamp *observed inside a split* (e.g., `train_df["time"].max()`) can appear a few minutes earlier than the boundary cutoff timestamp.
  - This difference reflects sampling alignment rather than an error in the split logic.


## Step 9 - Standardise value using training split

### 9A) Fit a StandardScaler on the CPU dataset training values and store parameters

In [27]:
from sklearn.preprocessing import StandardScaler

# Initialise the scaler
scaler = StandardScaler()

# Fit the scaler using ONLY the training split (raw trip counts)
scaler.fit(train_df[["value"]])

# Store the training mean and standard deviation for documentation/reproducibility
cpu_scaler_parameters = {
    "mean": float(scaler.mean_[0]),
    "std": float(scaler.scale_[0]),
}

cpu_scaler_parameters


{'mean': 37.63693432781737, 'std': 14.731908616695504}

### 9B) Apply scaling to full dataframe and check training behaviour

In [28]:
# Apply the fitted scaler to the FULL CPU dataframe (train, validation, test)
df["value_scaled"] = scaler.transform(df[["value"]])

# Describe the scaled values on the training split to confirm mean ~ 0 and std ~ 1
scaled_training_summary = (
    df[df["split"] == "train"]["value_scaled"]
      .describe()[["mean", "std"]]
      .to_frame(name="Training value_scaled")
      .rename(index={"mean": "Mean", "std": "Std. deviation"})
      .round(4)
)

scaled_training_summary = scaled_training_summary.replace(-0.0, 0.0)

scaled_training_summary

,Training value_scaled
Mean,0.0
Std. deviation,1.0


### 9C) Summary of value_scaled by split

In [29]:
# Summary of scaled values per split
split_scaled_summary = (
    df.groupby("split")["value_scaled"]
      .describe()[["mean", "std", "min", "max"]]   # key statistics only
      .round(3)
      .reset_index()
)

split_scaled_summary


,split,mean,std,min,max
0,test,0.116,1.699,-1.772,4.233
1,train,-0.000,1.000,-0.654,4.233
2,validation,0.791,1.313,-1.605,4.233


## Reorder columns for a clean schema

In [30]:
column_order = [
    "time",
    "value",
    "value_scaled",
    "is_anomaly",
    "hour_of_day",
    "day_of_week",
    "is_weekend",
    "split",
    "case_study",
]

# Reorder the dataframe using the defined schema
df = df[column_order]

# Quick check of the first few rows
df.head()



,time,value,value_scaled,is_anomaly,hour_of_day,day_of_week,is_weekend,split,case_study
0,2014-05-14 01:14:00,85.835,3.271678,0,1,2,0,train,cpu
1,2014-05-14 01:19:00,88.167,3.429974,0,1,2,0,train,cpu
2,2014-05-14 01:24:00,44.595,0.472313,0,1,2,0,train,cpu
3,2014-05-14 01:29:00,56.282,1.265625,0,1,2,0,train,cpu
4,2014-05-14 01:34:00,36.534,-0.074867,0,1,2,0,train,cpu


## Recreate split-specific dataframes so they include `value_scaled`

In [31]:
train_df = df[df["split"] == "train"].copy()
validation_df = df[df["split"] == "validation"].copy()
test_df = df[df["split"] == "test"].copy()


In [32]:
#Quick check, rows should be equal
len(df), len(train_df) + len(validation_df) + len(test_df)

(18050, 18050)

## Step 10 – Save processed CPU utilisation datasets to disk

In [33]:
import os

# Folder for processed CPU taxi data 
output_dir = "../data/processed/cpu_utilisation"
os.makedirs(output_dir, exist_ok=True)

# Save full processed dataframe (ground truth processed file)
full_path = os.path.join(output_dir, "cpu_utilisation_full.csv")
df.to_csv(full_path, index=False)

# File paths for the three splits
train_path = os.path.join(output_dir, "cpu_utilisation_train.csv")
val_path   = os.path.join(output_dir, "cpu_utilisation_validation.csv")
test_path  = os.path.join(output_dir, "cpu_utilisationi_test.csv")

# Save each split without the index column
train_df.to_csv(train_path, index=False)
validation_df.to_csv(val_path, index=False)
test_df.to_csv(test_path, index=False)

train_path, val_path, test_path, full_path


('../data/processed/cpu_utilisation/cpu_utilisation_train.csv',
 '../data/processed/cpu_utilisation/cpu_utilisation_validation.csv',
 '../data/processed/cpu_utilisation/cpu_utilisationi_test.csv',
 '../data/processed/cpu_utilisation/cpu_utilisation_full.csv')

#### Save processed CPU utilisation datasets to disk

The processed CPU utilisation outputs are saved to the `data/processed/cpu_utilisation/` folder as four CSV files:

- `cpu_utilisation_full.csv`
  - Contains the complete CPU utilisation dataset after preprocessing.
  - Includes all rows and the standardised core columns used across case studies:
    - `time`, `value`, `value_scaled`, `is_anomaly`,
      `hour_of_day`, `day_of_week`, `is_weekend`,
      `split`, `case_study`.
  - Serves as the canonical processed reference file for the CPU case study and the source for any later cross-case stacking or analysis.

- `cpu_utilisation_train.csv`
  - Subset of `cpu_utilisation_full.csv` where `split == "train"`.
  - Used as the fitting-only segment for preprocessing statistics (for example, the scaling parameters) and for training models.

- `cpu_utilisation_validation.csv`
  - Subset of `cpu_utilisation_full.csv` where `split == "validation"`.
  - Used for model selection and tuning decisions, and for early evaluation checks before final testing.

- `cpu_utilisation_test.csv`
  - Subset of `cpu_utilisation_full.csv` where `split == "test"`.
  - Held-out data used for final evaluation and reporting of results.

All split-specific files are derived directly from the single processed full file, ensuring a consistent and reproducible link between the full processed dataset and the train/validation/test segments used in later experiments.
